# 09 非结构化文本：清洗、切分、去重

前面几节的表都是"一行一条、字段规整"的。但构建 SFT 数据集时，原料往往长这样：

- **长文档**：产品手册、Markdown、Word 导出的 txt；
- **网页**：正文里混着导航、脚本、广告、页脚；
- **日志/聊天记录**：一行一条，字段藏在文本里；
- **转载/重复内容**：同一篇文档在多个来源出现，措辞略有不同。

这些东西无法直接 `read_csv`，要先走一条固定管线：

```
抽取（HTML/MD/TXT → 纯文本）→ 清洗（噪声、空白、PII）→ 切分（chunk）→ 去重 → 质检
```

本节输出的 `chunks` 表就是下一步（10、12 节）构建 SFT 数据的原料。

> 环境：`llm_sft`，从 `python/pandas` 目录启动。

In [1]:
import hashlib
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup, Comment

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_colwidth", 80)

DATA = Path("data")
if not DATA.exists():
    DATA = Path("python/pandas/data")
DOCS = DATA / "docs"
OUT = DATA.parent / "output"
OUT.mkdir(exist_ok=True)

files = sorted(p for p in DOCS.iterdir() if p.is_file())
for p in files:
    print(f"{p.name:20s} {p.suffix:6s} {p.stat().st_size:>6,} 字节")

产品说明.md              .md     1,446 字节
客服问答.txt             .txt    1,129 字节
技术博客.html            .html   1,908 字节
转载_客服问答.txt          .txt      696 字节


## 1. 抽取：不同来源，不同抽法

| 来源 | 抽法 | 注意 |
| --- | --- | --- |
| `.txt` | 直接读 | 编码、页眉页脚、页码 |
| `.md` | 直接读，可选去掉 `#` 标记 | 代码块要保护，别被压成一行 |
| `.html` | `BeautifulSoup` 取正文 | 去掉 `script/style/nav/footer` 和注释 |
| `.pdf` | `pdfplumber` 等（本环境未装） | 分栏、页眉、表格容易乱 |

先看看原始文件有多"脏"：

In [2]:
raw_md = (DOCS / "产品说明.md").read_text(encoding="utf-8")
raw_txt = (DOCS / "客服问答.txt").read_text(encoding="utf-8")
raw_html = (DOCS / "技术博客.html").read_text(encoding="utf-8")

print("Markdown 原始开头:\n", raw_md[:100])
print("\n纯文本里的零宽字符数:", raw_txt.count("\u200b"), "| 全角空格数:", raw_txt.count("\u3000"))
print("\nHTML 里 script/style/nav 的数量:",
      {tag: len(BeautifulSoup(raw_html, "html.parser").find_all(tag)) for tag in ["script", "style", "nav", "footer"]})

Markdown 原始开头:
 # 智学助手产品说明

智学助手是一款面向企业培训场景的问答产品，支持文档导入、知识检索与多轮对话。
本文说明产品能力、接入方式与常见限制。

## 一、核心能力

- **文档问答**：支持 PDF

纯文本里的零宽字符数: 1 | 全角空格数: 22

HTML 里 script/style/nav 的数量: {'script': 1, 'style': 1, 'nav': 1, 'footer': 1}


In [3]:
# HTML → 纯文本：先删"非正文"的标签，再取文字
soup = BeautifulSoup(raw_html, "html.parser")
for tag in soup(["script", "style", "nav", "footer", "aside"]):
    tag.decompose()
for comment in soup.find_all(string=lambda s: isinstance(s, Comment)):
    comment.extract()

article = soup.find("article") or soup.body
html_text = article.get_text("\n")
print("HTML 正文前 200 字:\n", html_text[:200])

HTML 正文前 200 字:
 

大模型微调数据准备：从原始文档到 SFT 数据集


微调效果的上限由数据决定。本文按“抽取 → 清洗 → 切分 → 去重 → 质检 → 划分”的顺序，介绍一条可复现的数据管线。


1. 抽取


文档来源通常有 Markdown、纯文本、HTML 与 PDF。HTML 需要先移除导航、脚本、样式与页脚，只保留正文。


2. 清洗


常见的噪声包括：零宽字符、全角空格、重复页眉、页码行、


## 2. 清洗：一条一条加规则

这个语料里有五类噪声，逐条处理：

| 噪声 | 处理 |
| --- | --- |
| 零宽字符 `\u200b`、BOM | 删除 |
| 全角空格 `\u3000`、不换行空格 `\xa0` | 转普通空格 |
| 行内连续空格 / 制表符 | 压缩成一个 |
| 3 个以上连续换行 | 压成 2 个 |
| 页码行、重复页眉 | 按正则/关键词删掉 |

**特殊处理：代码块**。如果直接压缩空白，```` ```python ... ``` ```` 里的缩进会被毁掉，
所以先把代码块替换成占位符，清洗完再还原。

In [4]:
CODE_BLOCK = re.compile(r"```.*?```", re.S)


def protect_code(text: str) -> tuple[str, list[str]]:
    """把 ``` 围栏代码块换成占位符，返回 (处理后的文本, 代码块列表)。"""
    blocks: list[str] = []

    def repl(m):
        blocks.append(m.group(0))
        return f"\x00CODE{len(blocks) - 1}\x00"

    return CODE_BLOCK.sub(repl, text), blocks


def restore_code(text: str, blocks: list[str]) -> str:
    for i, block in enumerate(blocks):
        text = text.replace(f"\x00CODE{i}\x00", block)
    return text


PAGE_NUMBER = re.compile(r"^[—–\-]{0,2}\s*\d{1,3}\s*[—–\-]{0,2}$")
HEADING_MARK = re.compile(r"^#{1,6}\s+")


def clean_text(text: str, drop_headers: tuple[str, ...] = ()) -> str:
    text, blocks = protect_code(text)

    text = text.replace("\r\n", "\n")
    text = re.sub("[\u200b\u200c\u200d\ufeff]", "", text)          # 零宽字符
    text = text.replace("\u3000", " ").replace("\xa0", " ")        # 全角空格 / 不换行空格

    lines = []
    for line in text.split("\n"):
        line = re.sub(r"[ \t]+", " ", line).strip()
        if PAGE_NUMBER.match(line):                                 # 页码行
            continue
        if any(line.startswith(h) for h in drop_headers):            # 重复页眉
            continue
        line = HEADING_MARK.sub("", line)                            # 去掉 # 标题符号，保留标题文字
        lines.append(line)

    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return restore_code(text.strip(), blocks)


print("清洗前后对比")
print("-" * 50)
print("原文:", repr(raw_txt[:120]))
print("\n清洗后:", repr(clean_text(raw_txt, drop_headers=("客服问答手册",))[:120]))

清洗前后对比
--------------------------------------------------
原文: '客服问答手册（内部资料）\n\n\u3000\u3000第一章 退换货政策\u200b\n\n\u3000\u3000问：收到商品后发现质量问题怎么办？\n答：请在签收后 7 天内联系客服，提供订单号与问题照片，客服会在 24 小时内给出处理方案。\n属于质量问题的，运费由我们承担；非质量问题的，运费'

清洗后: '第一章 退换货政策\n\n问：收到商品后发现质量问题怎么办？\n答：请在签收后 7 天内联系客服，提供订单号与问题照片，客服会在 24 小时内给出处理方案。\n属于质量问题的，运费由我们承担；非质量问题的，运费由客户承担。\n\n问：可以无理由退货吗？'


### 关于 `NFKC`：全角转半角要小心

`unicodedata.normalize("NFKC", ...)` 能把 `ＡＢＣ１２３` 统一成 `ABC123`，但它**也会把中文标点转成半角**：
`，` → `,`、`：` → `:`。所以：

- 代码、URL、编号类字段：放心用 NFKC；
- 中文正文：如果下游要求保留中文标点，就别整段 NFKC。

In [5]:
sample = pd.Series(["ＡＢＣ１２３，测试。（括号）", "１２３－４５６"])
print("原文 :", sample.tolist())
print("NFKC :", sample.str.normalize("NFKC").tolist())

原文 : ['ＡＢＣ１２３，测试。（括号）', '１２３－４５６']
NFKC : ['ABC123,测试。(括号)', '123-456']


In [6]:
# PII 脱敏：训练数据里不该出现真实邮箱、手机号、身份证号
EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
PHONE = re.compile(r"\b1[3-9]\d{9}\b")
ID_CARD = re.compile(r"\b\d{6}(?:19|20)\d{2}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])\d{3}[\dXx]\b")


def mask_pii(text: str) -> str:
    text = EMAIL.sub("[EMAIL]", text)
    text = PHONE.sub("[手机号]", text)
    text = ID_CARD.sub("[身份证号]", text)
    return text


print(mask_pii("联系 zhang.wei@corp.example.com 或 13912345678，身份证 110101199003071234"))

联系 [EMAIL] 或 [手机号]，身份证 [身份证号]


## 3. 切分：决定模型能"看到"多少上下文

为什么要切：

- 模型有上下文预算（比如 8k token），一篇 1 万字的文档塞不进去；
- 检索（RAG）需要小片段才能精确命中；
- 训练样本长度最好集中在一个区间，太长浪费显存、太短学不到东西。

切分的三个原则：

1. **优先在语义边界切**：段落 → 句子（`。！？`）→ 最后才硬切；
2. **加重叠（overlap）**：相邻片段共享一小段，避免答案正好被切在边界；
3. **长度可控**：中文常用 300~800 字，重叠 10%~20%。

In [7]:
def chunk_text(text: str, size: int = 300, overlap: int = 60) -> list[str]:
    """滑动窗口切分：优先在句号/感叹号/问号/换行处断开，窗口之间重叠 overlap 个字符。"""
    text = text.strip()
    chunks: list[str] = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            window = text[start:end]
            cut = max(window.rfind(p) for p in "。！？\n")   # 找最靠后的句子边界
            if cut > size * 0.5:                            # 边界在后半段才采用，否则硬切
                end = start + cut + 1
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)               # 回退 overlap，避免死循环
    return chunks


demo_chunks = chunk_text(clean_text(raw_md), size=300, overlap=60)
print(f"切成 {len(demo_chunks)} 段，长度分别是 {[len(c) for c in demo_chunks]}")
print("\n第 1 段:\n", demo_chunks[0][:160])
print("\n第 2 段开头（与上一段有重叠）:\n", demo_chunks[1][:80])

切成 3 段，长度分别是 [288, 280, 256]

第 1 段:
 智学助手产品说明

智学助手是一款面向企业培训场景的问答产品，支持文档导入、知识检索与多轮对话。
本文说明产品能力、接入方式与常见限制。

一、核心能力

- **文档问答**：支持 PDF、Markdown、Word 三类文档导入，自动切分为片段并建立向量索引。
- **多轮对话**：保留最近 10 轮上下文，超出部

第 2 段开头（与上一段有重叠）:
 部署两种。云端 API 使用 Bearer Token 鉴权：

```python
import requests

resp = requests.post


In [8]:
# 所有文档走一遍完整流程，得到一张 chunk 表
def load_document(path: Path) -> str:
    if path.suffix == ".html":
        soup = BeautifulSoup(path.read_text(encoding="utf-8"), "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "aside"]):
            tag.decompose()
        for comment in soup.find_all(string=lambda s: isinstance(s, Comment)):
            comment.extract()
        body = soup.find("article") or soup.body
        return body.get_text("\n")
    return path.read_text(encoding="utf-8")


rows = []
for path in files:
    text = clean_text(load_document(path), drop_headers=("客服问答手册",))
    for i, chunk in enumerate(chunk_text(text)):
        rows.append({
            "doc_id": path.stem,
            "doc_type": path.suffix.lstrip("."),
            "chunk_id": i,
            "text": chunk,
            "n_chars": len(chunk),
        })
chunks = pd.DataFrame(rows)
print("总 chunk 数:", len(chunks))
chunks.groupby("doc_id")["n_chars"].agg(["count", "min", "max", "sum"]).sort_values("count", ascending=False)

总 chunk 数: 8


,count,min,max,sum
doc_id,,,,
产品说明,3,256,288,824
客服问答,2,152,289,441
技术博客,2,173,277,450
转载_客服问答,1,233,233,233


## 4. 去重：精确重复 + 近似重复

| 类型 | 判定 | 成本 | 工具 |
| --- | --- | --- | --- |
| 完全重复 | 归一化后字节完全一致 | 低 | `md5` / `hash` |
| 近似重复 | 内容高度相似（转载、改标题、加前后缀） | 中 | SimHash / MinHash |

去重不是"锦上添花"：训练集里同一段话重复 100 次，模型就会背下来；验证集和训练集撞车，指标会虚高。

In [9]:
def normalize_for_dedup(text: str) -> str:
    """去重前的归一化：去掉空白、标点、大小写差异，避免"看起来一样"的漏网。"""
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[\s\u3000]+", "", text)
    return re.sub(r"[^\w\u4e00-\u9fff]", "", text)


def exact_hash(text: str) -> str:
    return hashlib.md5(normalize_for_dedup(text).encode("utf-8")).hexdigest()


chunks["exact_hash"] = chunks["text"].map(exact_hash)
print("完全重复的 chunk 数:", chunks["exact_hash"].duplicated().sum())
chunks[chunks["exact_hash"].duplicated(keep=False)].sort_values("exact_hash")[["doc_id", "chunk_id", "n_chars"]]

完全重复的 chunk 数: 0


,doc_id,chunk_id,n_chars


In [10]:
def simhash(text: str, bits: int = 64) -> int:
    """把文本压成一个 64 位指纹：相似文本的指纹只差几位。"""
    tokens = re.findall(r"[\u4e00-\u9fff]|[a-zA-Z]+", text)
    vec = np.zeros(bits)
    for token in tokens:
        h = int(hashlib.md5(token.encode("utf-8")).hexdigest(), 16)
        for i in range(bits):
            vec[i] += 1 if (h >> i) & 1 else -1
    return int("".join("1" if v > 0 else "0" for v in vec), 2)


def hamming(a: int, b: int) -> int:
    return bin(a ^ b).count("1")


chunks["simhash"] = chunks["text"].map(simhash)

pairs = []
for i in range(len(chunks)):
    for j in range(i + 1, len(chunks)):
        pairs.append({
            "i": i, "j": j,
            "hamming": hamming(chunks["simhash"][i], chunks["simhash"][j]),
            "doc_i": chunks["doc_id"][i], "doc_j": chunks["doc_id"][j],
        })
pairs = pd.DataFrame(pairs).sort_values("hamming")
pairs.head(6)

,i,j,hamming,doc_i,doc_j
21,3,7,10,客服问答,转载_客服问答
4,0,5,19,产品说明,技术博客
18,3,4,19,客服问答,客服问答
0,0,1,20,产品说明,产品说明
7,1,2,21,产品说明,产品说明
3,0,4,21,产品说明,客服问答


In [11]:
# 汉明距离越小越像。阈值 12（64 位里差 12 位）在本例中能抓住"转载版"
THRESHOLD = 12
near_dup = pairs[pairs["hamming"] <= THRESHOLD]
print(f"距离 ≤ {THRESHOLD} 的近似重复对:", len(near_dup))
for _, row in near_dup.iterrows():
    print(f"  {row['doc_i']} 第 {chunks['chunk_id'][row['i']]} 段  ≈  {row['doc_j']} 第 {chunks['chunk_id'][row['j']]} 段  (距离 {row['hamming']})")
    print("  A:", chunks["text"][row["i"]][:50].replace("\n", " "))
    print("  B:", chunks["text"][row["j"]][:50].replace("\n", " "))

距离 ≤ 12 的近似重复对: 1
  客服问答 第 0 段  ≈  转载_客服问答 第 0 段  (距离 10)
  A: 第一章 退换货政策  问：收到商品后发现质量问题怎么办？ 答：请在签收后 7 天内联系客服，提供订单
  B: 第一章 退换货政策  问：收到商品后发现质量问题怎么办？ 答：请在签收后 7 天内联系客服，提供订单


In [12]:
# 贪心去重：按顺序保留，与已保留内容太像的丢弃
kept: list[int] = []
dropped: list[dict] = []
for i in range(len(chunks)):
    hit = next((k for k in kept if hamming(chunks["simhash"][i], chunks["simhash"][k]) <= THRESHOLD), None)
    if hit is None:
        kept.append(i)
    else:
        dropped.append({"丢弃": i, "与谁重复": hit, "doc": chunks["doc_id"][i], "距离": hamming(chunks["simhash"][i], chunks["simhash"][hit])})

print("保留:", len(kept), "| 丢弃:", len(dropped))
pd.DataFrame(dropped)

保留: 7 | 丢弃: 1


,丢弃,与谁重复,doc,距离
0,7,3,转载_客服问答,10


## 5. 质检：用表回答"这批数据能不能用"

至少看四个指标：

1. **长度分布**：是不是集中在合理区间，有没有超长/超短；
2. **语言比例**：中文占比，混进英文/乱码的片段要挑出来；
3. **符号比例**：表情、特殊符号太多通常是网页噪声；
4. **重复率**：精确/近似重复各占多少。

In [13]:
def text_stats(text: str) -> pd.Series:
    n = max(len(text), 1)
    zh = len(re.findall(r"[\u4e00-\u9fff]", text))
    symbols = sum(1 for ch in text if unicodedata.category(ch)[0] in {"S", "C"})
    return pd.Series({
        "n_chars": len(text),
        "zh_ratio": round(zh / n, 3),
        "symbol_ratio": round(symbols / n, 3),
        "n_lines": text.count("\n") + 1,
    })


quality = pd.concat([chunks[["doc_id", "chunk_id"]], chunks["text"].apply(text_stats)], axis=1)
quality[["n_chars", "zh_ratio", "symbol_ratio", "n_lines"]].describe().round(2)

,n_chars,zh_ratio,symbol_ratio,n_lines
count,8.0,8.00,8.00,8.00
mean,243.5,0.55,0.07,14.88
std,53.6,0.19,0.01,2.85
min,152.0,0.17,0.05,10.00
25%,218.0,0.50,0.06,12.75
50%,266.5,0.54,0.07,16.00
75%,282.0,0.62,0.07,17.00
max,289.0,0.78,0.09,18.00


In [14]:
# 把质检结论变成"可执行的过滤条件"
final = chunks.loc[kept].copy()
final = final[final["n_chars"] >= 80]                      # 太短的信息量不足
final = final[final["text"].map(lambda t: len(re.findall(r"[\u4e00-\u9fff]", t)) / max(len(t), 1) >= 0.3)]  # 中文占比过低
final = final[~final["text"].duplicated()]                  # 双保险：完全重复

print("原始 chunk:", len(chunks), "→ 最终:", len(final))
final[["doc_id", "chunk_id", "n_chars", "text"]].head(4)

原始 chunk: 8 → 最终: 6


,doc_id,chunk_id,n_chars,text
0,产品说明,0,288,智学助手产品说明\n\n智学助手是一款面向企业培训场景的问答产品，支持文档导入、知识检索与多轮对话。\n本文说明产品能力、接入方式与常见限制。\n\n一...
2,产品说明,2,256,t(resp.json())\n```\n\n注意：上面的代码块里有缩进和空行，清洗文本时不能被压成一行。\n\n三、常见限制\n\n1. 单次上传文档...
3,客服问答,0,289,第一章 退换货政策\n\n问：收到商品后发现质量问题怎么办？\n答：请在签收后 7 天内联系客服，提供订单号与问题照片，客服会在 24 小时内给出处理方...
4,客服问答,1,152,箱。\n\n问：保修期多长？\n答：数码类商品保修 12 个月，家居类 24 个月，以订单页面标注为准。\n\n附录 联系方式\n\n客服邮箱：serv...


In [15]:
# 存成 Parquet，供 10 / 12 节和后续训练脚本使用
final[["doc_id", "doc_type", "chunk_id", "text", "n_chars"]].to_parquet(OUT / "doc_chunks.parquet", index=False)
print("已写出:", (OUT / "doc_chunks.parquet").resolve())
pd.read_parquet(OUT / "doc_chunks.parquet").groupby("doc_id").size()

已写出: D:\Repository\Jianyu-is-learning\python\pandas\output\doc_chunks.parquet


doc_id
产品说明    2
客服问答    2
技术博客    2
dtype: int64

## 6. 小练习

1. 把 `chunk_text` 的 `size` 改成 150，看看 chunk 数和平均长度怎么变（提示：用 `agg` 统计）；
2. 加一条过滤规则：丢掉"问：……答：……"之外、且不含中文标点的片段；
3. 统计每个文档被丢弃的比例，判断哪份文档噪声最大。

In [16]:
for size in [150, 300, 600]:
    n = sum(len(chunk_text(clean_text(load_document(p)), size=size, overlap=size // 5)) for p in files)
    print(f"size={size:3d} → 共 {n} 段")

size=150 → 共 16 段
size=300 → 共 8 段
size=600 → 共 5 段


In [17]:
# 各文档在去重前后的变化
before = chunks.groupby("doc_id").size().rename("去重前")
after = final.groupby("doc_id").size().rename("最终")
pd.concat([before, after], axis=1).fillna(0).astype(int).assign(
    保留率=lambda d: (d["最终"] / d["去重前"]).round(2)
)

,去重前,最终,保留率
doc_id,,,
产品说明,3,2,0.67
客服问答,2,2,1.00
技术博客,2,2,1.00
转载_客服问答,1,0,0.00


## 7. 小结

**管线五步**

```python
text  = load_document(path)          # 1. 抽取（HTML 去标签）
text  = clean_text(text)             # 2. 清洗（空白、零宽、页码、代码块保护）
parts = chunk_text(text)             # 3. 切分（句边界 + 重叠）
parts = dedup(parts)                 # 4. 去重（hash + simhash）
parts = quality_filter(parts)        # 5. 质检（长度、语言、符号、重复）
```

**四条经验**

1. 清洗规则要**按语料定制**（页眉长什么样，只有你知道），不要指望一个万能函数。
2. 代码块、表格、公式要先保护起来再压缩空白，否则内容会被毁掉。
3. 去重必须在**切分之后、划分数据集之前**做，否则同一段话会同时出现在训练集和验证集。
4. 每个过滤规则都要记下"丢了多少"，否则数据出了问题无法回溯。

下一节：把这些 chunk 变成真正的 SFT 训练样本，见 `10_SFT数据集构建_JSONL与对话格式.ipynb`。